<a href="https://colab.research.google.com/github/EhsanGhasemi423/INFS-8368/blob/main/RNN_P6_BPTT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Backpropagation Through Time (BPTT)

So far, we have seen how an RNN processes a sequence **forward through time**.

Now we need to look more closely at **how an RNN actually learns from a sequence**.

### From Backpropagation to BPTT

In a regular neural network, **backpropagation** moves backward through the network to determine how the model parameters contributed to the prediction error.

An RNN introduces an additional dimension: **time**.

As the RNN processes a sequence, each hidden state depends on the previous hidden state:

```text
x₁        x₂        x₃        x₄
│         │         │         │
▼         ▼         ▼         ▼
h₁ ─────► h₂ ─────► h₃ ─────► h₄
```

We can think of this as **unrolling the RNN across time**.

Importantly, these are not four different RNNs.

```text
h₁        h₂        h₃        h₄
│         │         │         │
└─────────┴─────────┴─────────┘
       same RNN parameters
```

The **same weights are reused at every time step**.

During training, the prediction error must therefore be propagated backward through these connected time steps:

```text
Forward:

h₁ ─────► h₂ ─────► h₃ ─────► h₄ ─────► Loss


Backward:

h₁ ◄───── h₂ ◄───── h₃ ◄───── h₄ ◄───── Loss
                 gradients
```

This is called **Backpropagation Through Time (BPTT)**.







### Why Does This Need Special Attention?

For a short sequence, backpropagating through a few time steps is manageable.

But real sequences can contain hundreds or thousands of tokens:

```text
h₁ → h₂ → h₃ → ............... → h₁₀₀₀
```

The forward computation must pass through many recurrent steps, and training must propagate gradients back through those steps.

This raises two important questions:

1. **What happens to the gradient as it travels through many time steps?**
2. **Do we really need to backpropagate through the entire sequence?**

The following subsections examine these questions and explain why long sequences can make RNNs difficult to train.

> **Key idea:** BPTT is ordinary backpropagation applied to an RNN **unrolled across time**. The important complication is that the same parameters are reused across many connected time steps.

## Analysis of Gradients in RNNs

At each time step, an RNN calculates a new hidden state using the **current input** and the **previous hidden state**:

$$
h_t = f(x_t, h_{t-1}, w_h)
$$

where:

- $x_t$ = input at the current time step
- $h_{t-1}$ = hidden state (memory) from the previous time step
- $w_h$ = recurrent weights shared across time steps
- $h_t$ = new hidden state

So this formula simply says:

**current input + previous memory → RNN → new memory**

The RNN then uses the hidden state to produce an output:

$$
o_t = g(h_t, w_o)
$$

where $w_o$ represents the output-layer weights.

The loss is calculated across all $T$ time steps:

$$
L = \frac{1}{T}\sum_{t=1}^{T} l(y_t,o_t)
$$

This simply means:

**calculate the prediction error at each time step and take the average.**


### Why is backpropagation more difficult in an RNN?

The hidden states are connected through time:

**$h_1 \rightarrow h_2 \rightarrow h_3 \rightarrow \cdots \rightarrow h_T$**

Therefore, during **Backpropagation Through Time (BPTT)**, the gradient must travel backward through these connections.

For the recurrent weights $w_h$:

$$
\frac{\partial h_t}{\partial w_h}
=
\frac{\partial f}{\partial w_h}
+
\frac{\partial f}{\partial h_{t-1}}
\frac{\partial h_{t-1}}{\partial w_h}
$$

This formula says that the recurrent weights affect the current hidden state in **two ways**:

- $\frac{\partial f}{\partial w_h}$ → **direct effect:** how the weights affect the current hidden state
- $\frac{\partial f}{\partial h_{t-1}}\frac{\partial h_{t-1}}{\partial w_h}$ → **past effect:** how the weights affected the previous hidden state, which then affects the current hidden state



In simple terms:

**gradient = current effect + effect carried forward from previous time steps**

Because $h_{t-1}$ depends on $h_{t-2}$, and $h_{t-2}$ depends on $h_{t-3}$, the chain rule continues backward:

$$
\frac{\partial h_t}{\partial h_{t-1}}
\frac{\partial h_{t-1}}{\partial h_{t-2}}
\frac{\partial h_{t-2}}{\partial h_{t-3}}
\cdots
$$

This formula simply shows that the gradient is **multiplied repeatedly as it travels backward through time**.





For long sequences, these repeated multiplications can create two problems:

```text
many time steps
      ↓
many derivative multiplications
      ↓
 ┌──────────────────┬──────────────────┐
 ↓                                      ↓
gradient becomes small             gradient becomes large
 ↓                                      ↓
Vanishing Gradient                  Exploding Gradient
```

If the derivatives are repeatedly **smaller than 1**:

$$
0.8 \times 0.8 \times 0.8 \times \cdots \rightarrow 0
$$

the gradient becomes very small → **Vanishing Gradient**

If the derivatives are repeatedly **larger than 1**:

$$
1.5 \times 1.5 \times 1.5 \times \cdots \rightarrow \text{very large}
$$

the gradient becomes very large → **Exploding Gradient**

> **Key idea:** BPTT uses the same chain rule as ordinary backpropagation, but applies it **backward through time**. Because derivatives are repeatedly multiplied across many time steps, gradients can become extremely small or extremely large.

## Strategies for Backpropagation Through Time

Because gradients in an RNN can travel backward through many time steps, we must decide **how far back to compute the gradient**.

### 1. Full Computation

With **full BPTT**, we backpropagate through the entire sequence.

```text
h₁ → h₂ → h₃ → h₄ → h₅ → ... → hₜ
↑_________________________________|
          gradient travels
          all the way back
```

**Advantage:** Uses information from the entire sequence.

**Problems:**

- expensive for long sequences,
- requires more memory,
- creates a very long chain of derivative multiplications,
- increases the risk of **vanishing or exploding gradients**.

For these reasons, full computation can become impractical for very long sequences.

---


### 2. Truncated Backpropagation Through Time

Instead of going all the way back, we stop the gradient after a fixed number of time steps, usually denoted by $\tau$.

For example, if:

$$
\tau = 3
$$

the gradient only travels through the most recent 3 steps:

```text
h₁ → h₂ → h₃ → h₄ → h₅ → h₆
               ↑____________|
               backpropagate
                3 steps only
```

Mathematically, instead of following the gradient through the entire history,

$$
h_t \rightarrow h_{t-1} \rightarrow h_{t-2} \rightarrow \cdots \rightarrow h_1
$$

we stop after $\tau$ steps:

$$
h_t \rightarrow h_{t-1} \rightarrow \cdots \rightarrow h_{t-\tau}
$$

This is called **Truncated Backpropagation Through Time (TBPTT)**.




**Advantages:**

- faster computation,
- lower memory requirements,
- shorter and more stable gradient paths.

**Trade-off:** The gradient is now an **approximation** because very old time steps are ignored.

> In practice, truncated BPTT is commonly preferred because it provides a practical balance between learning from previous time steps and keeping training manageable.

---



### 3. Randomized Truncation

Instead of always stopping BPTT after a fixed number of steps, **randomized truncation randomly decides whether the gradient stops or continues backward**.

The random variable $\xi_t$ controls this:

- $\xi_t = 0$ → **stop** the backward path
- $\xi_t \neq 0$ → **continue** backward

$$
z_t =
\frac{\partial f}{\partial w_h}
+
\xi_t
\frac{\partial f}{\partial h_{t-1}}
\frac{\partial h_{t-1}}{\partial w_h}
$$

When (“ksee”) $\xi_t=0$, the **past-effect term disappears**, so backpropagation stops at that time step.

> **Key idea:** Randomized truncation reduces the cost of full BPTT while sometimes allowing gradients to travel farther back in time.

### Comparing BPTT Strategies

There are three ways we could propagate gradients through an RNN sequence:

| Strategy | Idea | Practical? |
|---|---|---|
| **Full BPTT** | Backpropagate through the entire sequence | ❌ Expensive and unstable for long sequences |
| **Randomized Truncation** | Randomly choose how far gradients travel backward | ⚠️ Possible, but rarely provides much benefit |
| **Truncated BPTT** | Backpropagate a fixed number of time steps | ✅ Common practical approach |

In our RNN example, we used:

`num_steps = 32`

so the model processes the text in sequences of **32 time steps** rather than backpropagating through the entire text.

> **Key idea:** Truncated BPTT is a practical compromise; it gives the RNN some historical context while keeping computation manageable and gradients more stable.

## Summary

- **BPTT** is ordinary backpropagation applied to an RNN **across time steps**.
- Because recurrent connections are reused over time, gradients must travel backward through many hidden states.
- Over long sequences, gradients may:
  - **vanish** → become too small, making long-term relationships difficult to learn.
  - **explode** → become too large, making training unstable.

- **Truncated BPTT** limits how far gradients travel backward, making training more practical.

### Why Do We Need Something Better?

A simple RNN has difficulty remembering information over **long periods of time** because the influence of earlier information can gradually disappear.

This leads to an important question:

> **Can we design an RNN that decides what information to remember, what to forget, and what new information to store?**

That is exactly the idea behind **Long Short-Term Memory (LSTM)** networks.

➡️ **Next: LSTM — giving RNNs a better mechanism for learning long-term dependencies.**